In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Setup paths
from pathlib import Path
from zipfile import ZipFile
import os

# Source ZIPs in Drive
DRIVE = Path("/content/drive/MyDrive/deepfake_project")
ZIP_TRAIN = DRIVE / "zips/dataset_train.zip"
ZIP_TEST  = DRIVE / "zips/dataset_test.zip"

# Destination folders in Colab
DST_AFHQ_TRAIN   = Path("/content/stargan-v2/combined_balanced/train")
DST_AFHQ_TEST    = Path("/content/stargan-v2/combined_balanced/test")



# Create destinations
DST_AFHQ_TRAIN.mkdir(parents=True, exist_ok=True)
DST_AFHQ_TEST.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def safe_extract_images(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue
            # Only extract images
            suffix = Path(info.filename).suffix.lower()
            if suffix not in IMG_EXT:
                continue

            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

# 3) Extract train + test images into /content/stargan-v2/combined_balanced
n_train = safe_extract_images(ZIP_TRAIN, DST_AFHQ_TRAIN)
n_test  = safe_extract_images(ZIP_TEST,  DST_AFHQ_TEST)


print(f"[DONE] Extracted TRAIN images -> {DST_AFHQ_TRAIN}: {n_train} files")
print(f"[DONE] Extracted TEST  images -> {DST_AFHQ_TEST}: {n_test} files")

Mounted at /content/drive
[DONE] Extracted TRAIN images -> /content/stargan-v2/combined_balanced/train: 22572 files
[DONE] Extracted TEST  images -> /content/stargan-v2/combined_balanced/test: 9672 files


In [ ]:
# ===========================
# PATCH: Artifact-writing pipeline (TRAIN -> APPLY TO TEST)
# Creates forensic features, BoVW, IDF, PCA, and KMeans cluster labels.
# Writes outputs into /content/stargan-v2/bovw in the exact naming patterns your EDA expects.
# ===========================
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import math
from joblib import dump, load
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

BASE = Path("/content/stargan-v2")
DATA_ROOT = BASE / "combined_balanced"     # contains train/ and test/
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT  = DATA_ROOT / "test"

BOVW_DIR = BASE / "bovw"
BOVW_DIR.mkdir(parents=True, exist_ok=True)

RNG = 1337
np.random.seed(RNG)

print("BASE:", BASE)
print("TRAIN_ROOT exists:", TRAIN_ROOT.exists(), TRAIN_ROOT)
print("TEST_ROOT exists:", TEST_ROOT.exists(), TEST_ROOT)
print("BOVW_DIR:", BOVW_DIR)


BASE: /content/stargan-v2
TRAIN_ROOT exists: True /content/stargan-v2/combined_balanced/train
TEST_ROOT exists: True /content/stargan-v2/combined_balanced/test
BOVW_DIR: /content/stargan-v2/bovw


In [ ]:
# -----------------------------
# BUILD IMAGE INDEX (train/test) with labels and technique names
# AUDIT NOTE:
# - Paths are explicitly sorted for deterministic row order across runs.
# - A stable relative_path column is added for transparent downstream alignment checks.
# -----------------------------
import re

def _natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]


def list_images(root: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    return sorted(files, key=lambda p: _natural_key(str(p.relative_to(root)).replace("\\", "/")))


def parse_label_and_tech(p: Path):
    parts = p.parts
    split = "train" if "train" in parts else ("test" if "test" in parts else "unknown")
    if "real" in parts:
        label = 0
        tech = "real"
    elif "fake" in parts:
        label = 1
        i = parts.index("fake")
        tech = parts[i + 1] if i + 1 < len(parts) else "fake_unknown"
    else:
        label = None
        tech = "unknown"
    return split, label, tech


def infer_species(p: Path):
    species_vocab = {"cat", "dog", "wild"}
    for part in p.parts:
        token = str(part).lower()
        if token in species_vocab:
            return token
    name = p.name.lower()
    for s in sorted(species_vocab):
        if s in name:
            return s
    return ""


def build_df(split_root: Path):
    rows = []
    image_paths = list_images(split_root)
    for p in image_paths:
        split, label, tech = parse_label_and_tech(p)
        if label is None:
            continue
        rel = str(p.relative_to(split_root)).replace("\\", "/")
        rows.append({
            "path": str(p),
            "relative_path": rel,
            "filename": p.name,
            "split": split,
            "label": int(label),
            "label_name": "fake" if int(label) == 1 else "real",
            "species": infer_species(p),
            "fake_technique": tech if label == 1 else "",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No images found under: {split_root}")

    # Final deterministic order + audit index
    df = df.sort_values(["relative_path", "path"], kind="mergesort").reset_index(drop=True)
    df.insert(0, "row_id", np.arange(len(df), dtype=np.int64))

    if df["path"].duplicated().any():
        dupes = df.loc[df["path"].duplicated(), "path"].tolist()[:10]
        raise RuntimeError(f"Duplicate image paths detected under {split_root}: {dupes}")

    return df


df_train = build_df(TRAIN_ROOT)
df_test = build_df(TEST_ROOT)

print("Train counts:\n", df_train["label_name"].value_counts())
print("Test counts:\n", df_test["label_name"].value_counts())
print("Train fake techniques (top):\n", df_train.loc[df_train.label == 1, "fake_technique"].value_counts().head(20))

print("Train deterministic order sample:")
display(df_train[["row_id", "relative_path", "label_name", "species", "fake_technique"]].head(10))

print("Test deterministic order sample:")
display(df_test[["row_id", "relative_path", "label_name", "species", "fake_technique"]].head(10))

Train counts:
 label_name
fake    11286
real    11286
Name: count, dtype: int64
Test counts:
 label_name
fake    4836
real    4836
Name: count, dtype: int64
Train fake techniques (top):
 fake_technique
copy_move        1881
inpaint          1881
postproc         1881
splicing         1881
stargan_tiles    1881
swap_like        1881
Name: count, dtype: int64
Train deterministic order sample:


,row_id,relative_path,label_name,species,fake_technique
0,0,fake/copy_move/copy_move__00329a52__pixabay_do...,fake,dog,copy_move
1,1,fake/copy_move/copy_move__004af61d__pixabay_do...,fake,dog,copy_move
2,2,fake/copy_move/copy_move__008a6e40__flickr_wil...,fake,wild,copy_move
3,3,fake/copy_move/copy_move__00ae54fc__pixabay_ca...,fake,cat,copy_move
4,4,fake/copy_move/copy_move__00ef8c40__flickr_dog...,fake,dog,copy_move
5,5,fake/copy_move/copy_move__0106e9a5__pixabay_ca...,fake,cat,copy_move
6,6,fake/copy_move/copy_move__015a7099__pixabay_ca...,fake,cat,copy_move
7,7,fake/copy_move/copy_move__015e7ed8__flickr_cat...,fake,cat,copy_move
8,8,fake/copy_move/copy_move__01a4aa4f__flickr_cat...,fake,cat,copy_move
9,9,fake/copy_move/copy_move__01b82a99__flickr_dog...,fake,dog,copy_move


Test deterministic order sample:


,row_id,relative_path,label_name,species,fake_technique
0,0,fake/copy_move/copy_move__00a045b7__pixabay_ca...,fake,cat,copy_move
1,1,fake/copy_move/copy_move__00a80d4c__pixabay_ca...,fake,cat,copy_move
2,2,fake/copy_move/copy_move__00d19a01__pixabay_ca...,fake,cat,copy_move
3,3,fake/copy_move/copy_move__016f7d8d__pixabay_ca...,fake,cat,copy_move
4,4,fake/copy_move/copy_move__01d32f51__flickr_cat...,fake,cat,copy_move
5,5,fake/copy_move/copy_move__01f555ea__pixabay_ca...,fake,cat,copy_move
6,6,fake/copy_move/copy_move__02936c62__pixabay_do...,fake,dog,copy_move
7,7,fake/copy_move/copy_move__02d85e14__flickr_wil...,fake,wild,copy_move
8,8,fake/copy_move/copy_move__0343493b__flickr_dog...,fake,dog,copy_move
9,9,fake/copy_move/copy_move__03593484__flickr_wil...,fake,wild,copy_move


In [ ]:
import numpy as np
# -----------------------------
# FORENSIC FEATURE EXTRACTION (no CSV written here; wide CSV written later)
# -----------------------------
def lap_var(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def edge_density(gray):
    edges = cv2.Canny(gray, 50, 150)
    return float((edges > 0).mean())

def brightness(gray):
    return float(gray.mean())

def contrast(gray):
    return float(gray.std())

def sharpness_l1_mean(gray):
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.abs(gx) + np.abs(gy)
    return float(mag.mean())

def compute_features(path: str):
    p = Path(path)
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        return None
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return {
        "width": int(w),
        "height": int(h),
        "aspect_ratio": float(w / (h + 1e-12)),
        "brightness": brightness(gray),
        "contrast": contrast(gray),
        "lap_var": lap_var(gray),
        "edge_density": edge_density(gray),
        "sharpness_l1_mean": sharpness_l1_mean(gray),
        "file_size": int(p.stat().st_size),
    }

def add_features(df):
    feats = []
    for path in df["path"].tolist():
        f = compute_features(path)
        feats.append(f if f is not None else {
            "width": np.nan, "height": np.nan, "aspect_ratio": np.nan,
            "brightness": np.nan, "contrast": np.nan, "lap_var": np.nan,
            "edge_density": np.nan, "sharpness_l1_mean": np.nan, "file_size": np.nan
        })
    feat_df = pd.DataFrame(feats)
    out = pd.concat([df.reset_index(drop=True), feat_df], axis=1)
    if "label_name" not in out.columns:
        out["label_name"] = np.where(out["label"].values==1, "fake", "real")
    return out

df_train_feat = add_features(df_train)
df_test_feat  = add_features(df_test)

# NOTE: Do not write narrow image_features CSVs here.
# Wide feature tables are written later (after BoVW/PCA/KMeans) to:
#   - image_features.csv
#   - image_features_test.csv


In [ ]:
# -----------------------------
# BoVW + IDF + PCA (FIT ON TRAIN ONLY; APPLY TO TEST WITHOUT REFIT)
# -----------------------------
def get_desc_orb(path: str, nfeatures=2000):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    orb = cv2.ORB_create(nfeatures=nfeatures)
    kps, des = orb.detectAndCompute(img, None)
    if des is None or len(des)==0:
        return None
    return des.astype(np.float32)

def build_desc_list(paths):
    return [get_desc_orb(p) for p in paths]

def balanced_codebook_paths(df, n_real=800, n_per_tech=800, rng=1337):
    rs = np.random.RandomState(rng)
    paths = df["path"].values
    y = df["label"].values
    tech = df["fake_technique"].values
    idxs = []

    real_idx = np.where(y==0)[0]
    if len(real_idx):
        take = min(n_real, len(real_idx))
        idxs.append(rs.choice(real_idx, size=take, replace=False))

    fake_idx = np.where(y==1)[0]
    for t in np.unique(tech[fake_idx]):
        ti = np.where((y==1) & (tech==t))[0]
        if len(ti)==0:
            continue
        take = min(n_per_tech, len(ti))
        idxs.append(rs.choice(ti, size=take, replace=False))

    sel = np.unique(np.concatenate(idxs)) if idxs else np.array([], dtype=int)
    return paths[sel].tolist()

def fit_idf(H_train: np.ndarray) -> np.ndarray:
    N = H_train.shape[0]
    df = (H_train > 0).sum(axis=0)
    return (np.log((N+1)/(df+1)) + 1.0).astype(np.float32)

def apply_bovw(desc_list, codebook, K: int) -> np.ndarray:
    H = np.zeros((len(desc_list), K), dtype=np.float32)
    for i, des in enumerate(desc_list):
        if des is None or len(des)==0:
            continue
        words = codebook.predict(des)
        H[i] = np.bincount(words, minlength=K).astype(np.float32)
    return H

def bovw_normalize(H: np.ndarray, idf: np.ndarray) -> np.ndarray:
    # REQUIRED sequence: sqrt/Hellinger -> TF-IDF -> L2
    H = np.sqrt(H, dtype=np.float32)
    H = H * idf
    H = normalize(H, norm="l2")
    return H

# Choose fixed artifacts (use nested_cv_bovw() elsewhere to select these values)
K = 256
PCA_DIMS = 96
DET_NAME = "orb"

# 1) Fit codebook on balanced TRAIN subset (reduces technique bias)
cb_paths = balanced_codebook_paths(df_train_feat, rng=RNG)
cb_descs = build_desc_list(cb_paths)
X_code = np.vstack([d for d in cb_descs if d is not None and len(d)>0])
if X_code.shape[0] == 0:
    raise RuntimeError("No descriptors found for codebook fitting.")

codebook = MiniBatchKMeans(n_clusters=K, batch_size=4096, random_state=RNG, max_iter=100, n_init=3).fit(X_code)
dump(codebook, BOVW_DIR / f"codebook_{DET_NAME}_K{K}.joblib")

# 2) Build TRAIN BoVW histograms
train_paths = df_train_feat["path"].tolist()
train_descs = build_desc_list(train_paths)
H_train_raw = apply_bovw(train_descs, codebook, K)

# 3) Fit + save IDF (required for test normalization space)
idf = fit_idf(H_train_raw)
np.save(BOVW_DIR / f"idf_{DET_NAME}_K{K}.npy", idf)

# 4) Normalize train BoVW and save
H_train = bovw_normalize(H_train_raw, idf)
np.save(BOVW_DIR / f"H_bovw_{DET_NAME}_K{K}_train.npy", H_train)

# 5) Fit PCA on TRAIN only and save
pca = PCA(n_components=PCA_DIMS, random_state=RNG).fit(H_train)
Xp_train = pca.transform(H_train)
dump(pca, BOVW_DIR / f"pca_{DET_NAME}_K{K}_to{PCA_DIMS}.joblib")
np.save(BOVW_DIR / f"Xp_{DET_NAME}_K{K}_to{PCA_DIMS}_train.npy", Xp_train)

# 6) Apply to TEST without refitting
test_paths = df_test_feat["path"].tolist()
test_descs = build_desc_list(test_paths)
H_test_raw = apply_bovw(test_descs, codebook, K)

# Require IDF presence (fail fast if missing)
idf_path = BOVW_DIR / f"idf_{DET_NAME}_K{K}.npy"
if not idf_path.exists():
    raise FileNotFoundError(f"Missing required IDF: {idf_path}")
idf_loaded = np.load(idf_path)

H_test = bovw_normalize(H_test_raw, idf_loaded)
np.save(BOVW_DIR / f"H_bovw_{DET_NAME}_K{K}_test.npy", H_test)

Xp_test = pca.transform(H_test)
np.save(BOVW_DIR / f"Xp_{DET_NAME}_K{K}_to{PCA_DIMS}_test.npy", Xp_test)

print("Wrote BoVW artifacts for train+test into:", BOVW_DIR)


Wrote BoVW artifacts for train+test into: /content/stargan-v2/bovw


In [ ]:
# -----------------------------
# KMeans CLUSTERS (fit on TRAIN PCA; apply to TEST PCA)
# Writes km_img_labels_k10.npy / km_img_labels_k50.npy (TRAIN) + optional *_test.npy
# -----------------------------
from sklearn.preprocessing import normalize

Xn_train = normalize(Xp_train, norm="l2")
Xn_test  = normalize(Xp_test, norm="l2")

km2  = MiniBatchKMeans(n_clusters=2,  random_state=RNG, n_init=3, batch_size=2048, max_iter=100).fit(Xn_train)
km10 = MiniBatchKMeans(
    n_clusters=10,
    random_state=RNG,
    n_init=3,
    batch_size=2048,
    max_iter=100,
    init_size=3*2048  # optional, helps stability
).fit(Xn_train)

km50 = MiniBatchKMeans(
    n_clusters=50,
    random_state=RNG,
    n_init=3,
    batch_size=2048,
    max_iter=100,
    init_size=3*2048  # optional
).fit(Xn_train)

train_k2  = km2.predict(Xn_train)
test_k2   = km2.predict(Xn_test)

train_k10 = km10.predict(Xn_train)
train_k50 = km50.predict(Xn_train)
test_k10  = km10.predict(Xn_test)
test_k50  = km50.predict(Xn_test)

np.save(BOVW_DIR/"km_img_labels_k2.npy", train_k2)
np.save(BOVW_DIR/"km_img_labels_k2_test.npy", test_k2)
np.save(BOVW_DIR/"km_img_labels_k10.npy", train_k10)
np.save(BOVW_DIR/"km_img_labels_k50.npy", train_k50)
np.save(BOVW_DIR/"km_img_labels_k10_test.npy", test_k10)
np.save(BOVW_DIR/"km_img_labels_k50_test.npy", test_k50)

dump(km2,  BOVW_DIR/"km_k2.joblib")
dump(km10, BOVW_DIR/"km_k10.joblib")
dump(km50, BOVW_DIR/"km_k50.joblib")

# Attach cluster labels back into features CSVs
df_train_feat["km_label_k2"]  = train_k2
df_train_feat["km_label_k10"] = train_k10
df_train_feat["km_label_k50"] = train_k50
df_test_feat["km_label_k10"]  = test_k10
df_test_feat["km_label_k50"]  = test_k50
df_test_feat["km_label_k2"]  = test_k2


print("Attached cluster labels to df_train_feat / df_test_feat (wide CSVs written later).")


Attached cluster labels to df_train_feat / df_test_feat (wide CSVs written later).


In [ ]:
# -----------------------------
# NEW: Silhouette score grid (sampled; expensive on full data)
# - Saves to OUT_DIR/"silhouette_k_grid.csv" so EDA can find it.
# -----------------------------
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from pathlib import Path

OUT_DIR  = globals().get("OUT_DIR", BOVW_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use PCA features for clustering space (same as KMeans section)
X_base = Xp_train
X_base = normalize(X_base, norm="l2")

RNG_SIL = 42
MAX_SIL_SAMPLES = 4000  # keep small; silhouette is O(n^2) in worst case
rs = np.random.RandomState(RNG_SIL)

n = X_base.shape[0]
n_s = min(MAX_SIL_SAMPLES, n)
idx = rs.choice(n, size=n_s, replace=False) if n_s < n else np.arange(n)

X_s = X_base[idx]

k_list = [2, 5, 10, 20, 30, 40, 50]
rows = []

for k in k_list:
    if n_s <= k:
        rows.append({"k": k, "silhouette": np.nan, "n_sample": int(n_s), "random_state": RNG_SIL, "note": "insufficient_samples"})
        continue
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RNG_SIL,
        n_init=3,
        batch_size=2048,
        max_iter=100,
        init_size=min(3*2048, n_s)
    ).fit(X_s)
    labels = km.labels_
    # Guard: silhouette requires >=2 clusters present
    if len(np.unique(labels)) < 2:
        rows.append({"k": k, "silhouette": np.nan, "n_sample": int(n_s), "random_state": RNG_SIL, "note": "single_cluster"})
        continue
    sil = float(silhouette_score(X_s, labels, metric="euclidean"))
    rows.append({"k": k, "silhouette": sil, "n_sample": int(n_s), "random_state": RNG_SIL, "note": ""})

df_sil = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
sil_path = OUT_DIR / "silhouette_k_grid.csv"
df_sil.to_csv(sil_path, index=False)

print(f"Wrote: {sil_path}")
display(df_sil)

Wrote: /content/stargan-v2/bovw/silhouette_k_grid.csv


,k,silhouette,n_sample,random_state,note
0,2,0.142945,4000,42,
1,5,0.099495,4000,42,
2,10,0.073887,4000,42,
3,20,0.042437,4000,42,
4,30,0.014052,4000,42,
5,40,-0.000341,4000,42,
6,50,0.000739,4000,42,


In [ ]:

# -----------------------------
# NEW: Write full feature tables into image_features.csv and image_features_test.csv
# - Includes: metadata + forensic + KMeans labels + KMeans soft-prob features (k=2,10,50)
#            + BoVW histogram columns + PCA columns
# - Adds explicit ordering and audit columns for transparent downstream review.
# -----------------------------
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

OUT_DIR  = globals().get("OUT_DIR", BOVW_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _softmax_from_dist(D, tau=1.0):
    # D: (n_samples, k) distances to centroids (lower = closer)
    S = np.exp(-D / float(tau))
    return S / (S.sum(axis=1, keepdims=True) + 1e-12)


def _sha1_list(values):
    h = hashlib.sha1()
    for v in values:
        h.update(str(v).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()


# Ensure we have normalized PCA features used for clustering
from sklearn.preprocessing import normalize
Xn_train = normalize(Xp_train, norm="l2")
Xn_test  = normalize(Xp_test,  norm="l2")

# Soft probabilities (leakage-safe: km models fit on TRAIN only earlier in this notebook)
prob_k2_train  = _softmax_from_dist(km2.transform(Xn_train),  tau=1.0)
prob_k2_test   = _softmax_from_dist(km2.transform(Xn_test),   tau=1.0)
prob_k10_train = _softmax_from_dist(km10.transform(Xn_train), tau=1.0)
prob_k10_test  = _softmax_from_dist(km10.transform(Xn_test),  tau=1.0)
prob_k50_train = _softmax_from_dist(km50.transform(Xn_train), tau=1.0)
prob_k50_test  = _softmax_from_dist(km50.transform(Xn_test),  tau=1.0)

# Hard labels (needed by EDA + helpful for debugging)
labs_k2_train  = np.argmax(prob_k2_train,  axis=1).astype(np.int32)
labs_k2_test   = np.argmax(prob_k2_test,   axis=1).astype(np.int32)

# Save k=2 labels too (parity with k=10/k=50 label artifacts)
np.save(OUT_DIR/"km_img_labels_k2.npy", labs_k2_train)
np.save(OUT_DIR/"km_img_labels_k2_test.npy", labs_k2_test)

# Base header
base_cols = [
    "row_id", "split", "path", "relative_path", "filename", "label", "label_name", "species", "fake_technique",
    "width", "height", "aspect_ratio", "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var", "file_size",
    "bovw_K", "pca_dims", "km_k_best", "km_label_best",
    "km_label_k2", "km_label_k10", "km_label_k50"
]

def build_full_table(df_feat, H_norm, X_pca, labs_k2, labs_k10, labs_k50, prob_k2, prob_k10, prob_k50):
    if len(df_feat) != H_norm.shape[0] or len(df_feat) != X_pca.shape[0]:
        raise RuntimeError(
            f"Row alignment mismatch: len(df_feat)={len(df_feat)}, H_norm={H_norm.shape[0]}, X_pca={X_pca.shape[0]}"
        )

    # Define "best" KM as k=10 by default (consistent with earlier usage in your pipeline)
    KM_K_BEST = 10
    labs_best = labs_k10

    rel_paths = df_feat["relative_path"].astype(str).tolist() if "relative_path" in df_feat.columns else [""] * len(df_feat)
    filenames = df_feat["filename"].astype(str).tolist() if "filename" in df_feat.columns else [Path(p).name for p in df_feat["path"].tolist()]
    row_ids = df_feat["row_id"].to_numpy(dtype=np.int64) if "row_id" in df_feat.columns else np.arange(len(df_feat), dtype=np.int64)
    split_vals = df_feat["split"].astype(str).tolist() if "split" in df_feat.columns else [""] * len(df_feat)

    data = {
        "row_id": row_ids,
        "split": np.array(split_vals, dtype=object),
        "path": df_feat["path"].astype(str).values,
        "relative_path": np.array(rel_paths, dtype=object),
        "filename": np.array(filenames, dtype=object),
        "label": df_feat["label"].astype(np.int32).values,
        "label_name": df_feat["label_name"].astype(str).values,
        "species": df_feat["species"].fillna("").astype(str).values if "species" in df_feat.columns else np.array([""]*len(df_feat), dtype=object),
        "fake_technique": df_feat["fake_technique"].fillna("").astype(str).values if "fake_technique" in df_feat.columns else np.array([""]*len(df_feat), dtype=object),
        "width": df_feat["width"].values,
        "height": df_feat["height"].values,
        "aspect_ratio": df_feat["aspect_ratio"].values,
        "brightness": df_feat["brightness"].values,
        "contrast": df_feat["contrast"].values,
        "sharpness_l1_mean": df_feat["sharpness_l1_mean"].values,
        "edge_density": df_feat["edge_density"].values,
        "lap_var": df_feat["lap_var"].values,
        "file_size": df_feat["file_size"].values,
        "bovw_K": np.full((len(df_feat),), int(K), dtype=np.int32),
        "pca_dims": np.full((len(df_feat),), int(X_pca.shape[1]), dtype=np.int32),
        "km_k_best": np.full((len(df_feat),), int(KM_K_BEST), dtype=np.int32),
        "km_label_best": np.asarray(labs_best, dtype=np.int32),
        "km_label_k2": np.asarray(labs_k2, dtype=np.int32),
        "km_label_k10": np.asarray(labs_k10, dtype=np.int32),
        "km_label_k50": np.asarray(labs_k50, dtype=np.int32),
    }

    # prob features
    for j in range(prob_k2.shape[1]):
        data[f"prob_k2_{j}"] = prob_k2[:, j]
    for j in range(prob_k10.shape[1]):
        data[f"prob_k10_{j}"] = prob_k10[:, j]
    for j in range(prob_k50.shape[1]):
        data[f"prob_k50_{j}"] = prob_k50[:, j]

    # BoVW histogram columns
    for j in range(H_norm.shape[1]):
        data[f"bovw_{j}"] = H_norm[:, j]

    # PCA columns
    for j in range(X_pca.shape[1]):
        data[f"pca_{j}"] = X_pca[:, j]

    ordered_cols = (
        base_cols
        + [f"prob_k2_{j}" for j in range(prob_k2.shape[1])]
        + [f"prob_k10_{j}" for j in range(prob_k10.shape[1])]
        + [f"prob_k50_{j}" for j in range(prob_k50.shape[1])]
        + [f"bovw_{j}" for j in range(H_norm.shape[1])]
        + [f"pca_{j}" for j in range(X_pca.shape[1])]
    )
    out = pd.DataFrame(data, columns=ordered_cols)
    out = out.sort_values(["row_id", "relative_path", "path"], kind="mergesort").reset_index(drop=True)
    return out

# Build and write
train_full = build_full_table(
    df_train_feat,
    H_train,
    Xp_train,
    labs_k2=labs_k2_train,
    labs_k10=train_k10,
    labs_k50=train_k50,
    prob_k2=prob_k2_train,
    prob_k10=prob_k10_train,
    prob_k50=prob_k50_train
)
test_full = build_full_table(
    df_test_feat,
    H_test,
    Xp_test,
    labs_k2=labs_k2_test,
    labs_k10=test_k10,
    labs_k50=test_k50,
    prob_k2=prob_k2_test,
    prob_k10=prob_k10_test,
    prob_k50=prob_k50_test
)

# Final audit checks before write
if train_full["path"].duplicated().any() or test_full["path"].duplicated().any():
    raise RuntimeError("Duplicate paths detected in final full feature tables.")
if not np.array_equal(train_full["row_id"].to_numpy(), np.arange(len(train_full), dtype=np.int64)):
    raise RuntimeError("train_full row_id is not contiguous and ordered as expected.")
if not np.array_equal(test_full["row_id"].to_numpy(), np.arange(len(test_full), dtype=np.int64)):
    raise RuntimeError("test_full row_id is not contiguous and ordered as expected.")

train_csv_path = OUT_DIR / "image_features.csv"
test_csv_path  = OUT_DIR / "image_features_test.csv"
train_full.to_csv(train_csv_path, index=False)
test_full.to_csv(test_csv_path, index=False)

# Transparent row-order fingerprints for downstream verification
train_order_sha1 = _sha1_list(train_full["relative_path"].tolist())
test_order_sha1 = _sha1_list(test_full["relative_path"].tolist())
(OUT_DIR / "image_features_train_order_sha1.txt").write_text(train_order_sha1, encoding="utf-8")
(OUT_DIR / "image_features_test_order_sha1.txt").write_text(test_order_sha1, encoding="utf-8")

# Minimal schema report for reviewers
schema_report = pd.DataFrame({
    "column": train_full.columns,
    "dtype": [str(dt) for dt in train_full.dtypes]
})
schema_report.to_csv(OUT_DIR / "image_features_schema.csv", index=False)

print(f"Wrote: {train_csv_path} | Rows: {len(train_full)} | Cols: {train_full.shape[1]}")
print(f"Wrote: {test_csv_path}  | Rows: {len(test_full)}  | Cols: {test_full.shape[1]}")
print(f"Wrote: {OUT_DIR/'km_img_labels_k2.npy'} (train k=2 labels)")
print(f"Wrote: {OUT_DIR/'km_img_labels_k2_test.npy'} (test k=2 labels)")
print(f"Wrote: {OUT_DIR/'image_features_train_order_sha1.txt'} -> {train_order_sha1}")
print(f"Wrote: {OUT_DIR/'image_features_test_order_sha1.txt'}  -> {test_order_sha1}")
print(f"Wrote: {OUT_DIR/'image_features_schema.csv'}")


Wrote: /content/stargan-v2/bovw/image_features.csv | Rows: 22572 | Cols: 439
Wrote: /content/stargan-v2/bovw/image_features_test.csv  | Rows: 9672  | Cols: 439
Wrote: /content/stargan-v2/bovw/km_img_labels_k2.npy (train k=2 labels)
Wrote: /content/stargan-v2/bovw/km_img_labels_k2_test.npy (test k=2 labels)
Wrote: /content/stargan-v2/bovw/image_features_train_order_sha1.txt -> 0ce3639e73728f675ba3aacb71950f429591bb75
Wrote: /content/stargan-v2/bovw/image_features_test_order_sha1.txt  -> 6efd5435cf8fdd099f342ac068e606454925f6f3
Wrote: /content/stargan-v2/bovw/image_features_schema.csv


In [ ]:

# -----------------------------
# GUARDRAIL: verify expected artifacts exist and row alignment is transparent
# -----------------------------
must_exist = [
    "image_features.csv",
    "image_features_test.csv",
    f"H_bovw_{DET_NAME}_K{K}_train.npy",
    f"H_bovw_{DET_NAME}_K{K}_test.npy",
    f"Xp_{DET_NAME}_K{K}_to{PCA_DIMS}_train.npy",
    f"Xp_{DET_NAME}_K{K}_to{PCA_DIMS}_test.npy",
    f"pca_{DET_NAME}_K{K}_to{PCA_DIMS}.joblib",
    f"idf_{DET_NAME}_K{K}.npy",
    "km_img_labels_k2.npy",
    "km_img_labels_k10.npy",
    "km_img_labels_k50.npy",
    "image_features_train_order_sha1.txt",
    "image_features_test_order_sha1.txt",
    "image_features_schema.csv",
]
missing = [m for m in must_exist if not (BOVW_DIR/m).exists()]
if missing:
    raise RuntimeError(f"Missing expected BoVW/PCA/KMeans artifacts in {BOVW_DIR}: {missing}")

# Sanity-check final wide tables against in-memory dataframes
_guard_train = pd.read_csv(BOVW_DIR / "image_features.csv", low_memory=False)
_guard_test  = pd.read_csv(BOVW_DIR / "image_features_test.csv", low_memory=False)

if len(_guard_train) != len(df_train_feat):
    raise RuntimeError(f"Train CSV row count mismatch: csv={len(_guard_train)} vs df_train_feat={len(df_train_feat)}")
if len(_guard_test) != len(df_test_feat):
    raise RuntimeError(f"Test CSV row count mismatch: csv={len(_guard_test)} vs df_test_feat={len(df_test_feat)}")

if not np.array_equal(_guard_train["relative_path"].astype(str).to_numpy(), df_train_feat["relative_path"].astype(str).to_numpy()):
    raise RuntimeError("Train CSV relative_path order mismatch vs df_train_feat.")
if not np.array_equal(_guard_test["relative_path"].astype(str).to_numpy(), df_test_feat["relative_path"].astype(str).to_numpy()):
    raise RuntimeError("Test CSV relative_path order mismatch vs df_test_feat.")

print("✅ All expected BoVW/PCA/KMeans artifacts exist in:", BOVW_DIR)
print("✅ image_features.csv and image_features_test.csv row order matches the in-memory feature dataframes.")


✅ All expected BoVW/PCA/KMeans artifacts exist in: /content/stargan-v2/bovw
✅ image_features.csv and image_features_test.csv row order matches the in-memory feature dataframes.


In [ ]:

# -----------------------------
# AUDIT REPORT: transparent alignment and schema checks for reviewers
# -----------------------------
from pathlib import Path
import hashlib
import pandas as pd

OUT_DIR = globals().get("OUT_DIR", BOVW_DIR)
train_csv = OUT_DIR / "image_features.csv"
test_csv = OUT_DIR / "image_features_test.csv"

train_audit = pd.read_csv(train_csv, low_memory=False)
test_audit = pd.read_csv(test_csv, low_memory=False)

summary_rows = []
for split_name, frame in [("train", train_audit), ("test", test_audit)]:
    h = hashlib.sha1()
    for rp in frame["relative_path"].astype(str).tolist():
        h.update(rp.encode("utf-8")); h.update(b"\n")
    summary_rows.append({
        "split": split_name,
        "rows": int(len(frame)),
        "cols": int(frame.shape[1]),
        "unique_paths": int(frame["path"].nunique()),
        "unique_row_id": int(frame["row_id"].nunique()),
        "relative_path_sha1": h.hexdigest(),
        "min_row_id": int(frame["row_id"].min()),
        "max_row_id": int(frame["row_id"].max()),
    })

audit_df = pd.DataFrame(summary_rows)
audit_path = OUT_DIR / "program3_audit_report.csv"
audit_df.to_csv(audit_path, index=False)
print("Wrote:", audit_path)
display(audit_df)


Wrote: /content/stargan-v2/bovw/program3_audit_report.csv


,split,rows,cols,unique_paths,unique_row_id,relative_path_sha1,min_row_id,max_row_id
0,train,22572,439,22572,22572,0ce3639e73728f675ba3aacb71950f429591bb75,0,22571
1,test,9672,439,9672,9672,6efd5435cf8fdd099f342ac068e606454925f6f3,0,9671


In [ ]:
# ===========================
# PATCH: Auto-load TRAIN PCA features (Xp) if not already defined
# - Looks for either X_pca_*_train.npy OR Xp_*_train.npy under /content/stargan-v2/bovw
# - Creates Xp variable expected by downstream cells
# ===========================
from pathlib import Path
import numpy as np

BOVW_DIR = globals().get("BOVW_DIR", Path("/content/stargan-v2/bovw"))
OUT_DIR  = globals().get("OUT_DIR", BOVW_DIR)

def _pick_one(d: Path, pattern: str):
    cands = sorted(d.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None

if ("Xp" not in globals()) or (globals().get("Xp") is None):
    # Accept both historical naming conventions
    xp_path = _pick_one(BOVW_DIR, "X_pca_*_train.npy")
    if xp_path is None:
        xp_path = _pick_one(BOVW_DIR, "Xp_*_train.npy")
    if xp_path is None:
        raise RuntimeError(
            f"Could not find TRAIN PCA array in {BOVW_DIR}. Expected X_pca_*_train.npy or Xp_*_train.npy. "
            "Run the artifact pipeline cells first."
        )
    Xp = np.load(xp_path)
    print("Loaded Xp from:", xp_path.name, "shape=", Xp.shape)
else:
    print("Xp already defined; shape=", getattr(Xp, "shape", None))

# Optional: default X_for_km to Xp if not set
if ("X_for_km" not in globals()) or (globals().get("X_for_km") is None):
    X_for_km = Xp.copy()


Loaded Xp from: Xp_orb_K256_to96_train.npy shape= (22572, 96)


In [ ]:
# ===========================
# PATCH: Cluster-derived features (NO LEAKAGE)
# - Fits KMeans ONLY on each CV train fold, transforms val fold with same centroids
# - Z-scores distances using TRAIN stats only
# - Evaluates base vs base+dist vs base+soft assignments
# ===========================
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# Use the dataframe created by the artifact pipeline, otherwise fall back to df loaded from image_features.csv
feat_df = df_train_feat if "df_train_feat" in globals() else df

# Base tabular features you already use
extra_cols = ["brightness","contrast","sharpness_l1_mean","aspect_ratio","width","height",
              "edge_density","lap_var","file_size"]

extra = feat_df[extra_cols].to_numpy(dtype=np.float32)

# Xp should be the TRAIN PCA features (X_pca_*_train.npy). If you loaded under a different name, adjust.
if "Xp" not in globals() or Xp is None:
    raise RuntimeError("Xp (train PCA features) is not defined. Load X_pca_*_train.npy into Xp before running.")
X_base = np.hstack([Xp.astype(np.float32), extra])

y = feat_df["label"].to_numpy(dtype=int)

# X_for_km should be the normalized PCA space used for MiniBatchKMeans(often L2-normalized PCA features).
# If not defined elsewhere, default to L2-normalized Xp.
if "X_for_km" not in globals() or X_for_km is None:
    X_for_km = Xp.copy()
X_for_km = X_for_km.astype(np.float32)
# L2 normalize rows (safe even if already normalized)
row_norm = np.linalg.norm(X_for_km, axis=1, keepdims=True) + 1e-12
X_for_km = X_for_km / row_norm

def soft_assign(D, tau):
    S = np.exp(-D / float(tau))
    return S / (S.sum(axis=1, keepdims=True) + 1e-12)

def cv_auc_with_cluster_feats_noleak(X_base, X_norm, y, k_grid=(10,12,15,20,25,30), taus=(0.5,1.0,2.0), rng=1337):
    rows = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rng)

    for k in k_grid:
        for feat_kind in (["dist"] + [f"soft_tau={t}" for t in taus]):
            aucs = []

            for tr, va in skf.split(X_base, y):
                # Fit KMeans ONLY on train fold to avoid leakage
                km = MiniBatchKMeans(n_clusters=k, random_state=rng, n_init=3, batch_size=2048, max_iter=100).fit(X_norm[tr])

                d_tr = km.transform(X_norm[tr])
                d_va = km.transform(X_norm[va])

                # Z-score distances using TRAIN stats only
                mu = d_tr.mean(0)
                sd = d_tr.std(0) + 1e-8
                d_trn = (d_tr - mu) / sd
                d_van = (d_va - mu) / sd

                if feat_kind == "dist":
                    Xtr = np.hstack([X_base[tr], d_trn]).astype(np.float32)
                    Xva = np.hstack([X_base[va], d_van]).astype(np.float32)
                else:
                    tau = float(feat_kind.split("=")[1])
                    s_tr = soft_assign(d_tr, tau)
                    s_va = soft_assign(d_va, tau)
                    Xtr = np.hstack([X_base[tr], s_tr]).astype(np.float32)
                    Xva = np.hstack([X_base[va], s_va]).astype(np.float32)

                clf = Pipeline([
                    ("scaler", StandardScaler()),
                    ("lr", LogisticRegression(
                        solver="saga", penalty="l2", C=1.0, max_iter=3000,
                        random_state=rng, n_jobs=1
                    ))
                ])
                clf.fit(Xtr, y[tr])
                p = clf.predict_proba(Xva)[:, 1]
                aucs.append(roc_auc_score(y[va], p))

            rows.append({
                "k": int(k),
                "feat": feat_kind,
                "auc_mean": float(np.mean(aucs)),
                "auc_std": float(np.std(aucs, ddof=1)),
            })

    return pd.DataFrame(rows)

grid_df = cv_auc_with_cluster_feats_noleak(
    X_base, X_for_km, y,
    k_grid=(10,12,15,20,25,30),
    taus=(0.5,1.0,2.0),
    rng=RNG
)

grid_df = grid_df.sort_values(["auc_mean","feat"], ascending=[False, True])
print(grid_df.head(15).to_string(index=False))

grid_df.to_csv(OUT_DIR/"cluster_feature_grid_auc_NOLEAK.csv", index=False)
print("Wrote:", OUT_DIR/"cluster_feature_grid_auc_NOLEAK.csv")


 k         feat  auc_mean  auc_std
30 soft_tau=1.0  0.648122 0.005676
30 soft_tau=2.0  0.647961 0.005542
30 soft_tau=0.5  0.647873 0.005717
30         dist  0.647801 0.005629
25 soft_tau=1.0  0.647366 0.005968
25 soft_tau=2.0  0.647248 0.005903
25 soft_tau=0.5  0.646888 0.005862
25         dist  0.646805 0.005940
20 soft_tau=2.0  0.645687 0.005679
20 soft_tau=1.0  0.645605 0.005476
20         dist  0.645425 0.005571
20 soft_tau=0.5  0.645147 0.005211
15         dist  0.644844 0.005477
15 soft_tau=2.0  0.644685 0.005420
15 soft_tau=1.0  0.644331 0.005380
Wrote: /content/stargan-v2/bovw/cluster_feature_grid_auc_NOLEAK.csv


In [ ]:
# ==========================
# REQUIRED: descriptor extractor + cache for nested CV
# Fixes: NameError get_or_make_desc / det / det_name / desc_dim
# ==========================
from pathlib import Path
import hashlib
import numpy as np
import cv2

# Defaults (safe)
if "det_name" not in globals():
    det_name = "orb"
if "desc_dim" not in globals():
    desc_dim = 32  # ORB descriptor length
if "det" not in globals():
    det = None  # optional custom detector

# Cache directory
_DESC_CACHE_DIR = (Path(BOVW_DIR) if "BOVW_DIR" in globals() else Path(".")) / "desc_cache"
_DESC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _cache_key(img_path: Path, det_name: str, nfeatures: int) -> str:
    try:
        st = img_path.stat()
        sig = f"{img_path}|{st.st_size}|{int(st.st_mtime)}|{det_name}|{nfeatures}"
    except Exception:
        sig = f"{img_path}|{det_name}|{nfeatures}"
    return hashlib.sha1(sig.encode("utf-8")).hexdigest()

def _extract_orb(img_path: Path, nfeatures: int = 2000):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    orb = cv2.ORB_create(nfeatures=nfeatures)
    _, des = orb.detectAndCompute(img, None)
    if des is None or len(des) == 0:
        return None
    return des  # uint8

def get_or_make_desc(img_path: Path, det, det_name: str, desc_dim: int, nfeatures: int = 2000):
    """
    Returns local descriptors (N x desc_dim). Uses disk cache for speed.
    If det is provided and supports detectAndCompute, it will be used; otherwise ORB.
    """
    key = _cache_key(img_path, det_name, nfeatures)
    f = _DESC_CACHE_DIR / f"{key}.npy"
    if f.exists():
        arr = np.load(f, allow_pickle=False)
        if arr.size == 0:
            return None
        return arr

    if det is not None and hasattr(det, "detectAndCompute"):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        _, des = det.detectAndCompute(img, None)
        if des is None or len(des) == 0:
            return None
    else:
        des = _extract_orb(img_path, nfeatures=nfeatures)
        if des is None:
            return None

    np.save(f, des, allow_pickle=False)
    return des

print("Descriptor cache dir:", _DESC_CACHE_DIR)

Descriptor cache dir: /content/stargan-v2/bovw/desc_cache


In [ ]:
# ==========================
# PATCH: Leakage-free BoVW nested cross-validation (IDF fit inside inner folds)
# ==========================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def fit_idf(H_train: np.ndarray) -> np.ndarray:
    N = H_train.shape[0]
    df = (H_train > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1.0
    return idf.astype(np.float32)

def apply_bovw(desc_list, codebook, K: int) -> np.ndarray:
    H = np.zeros((len(desc_list), K), dtype=np.float32)
    for i, desc in enumerate(desc_list):
        if desc is None or len(desc) == 0:
            continue
        words = codebook.predict(desc)
        H[i] = np.bincount(words, minlength=K).astype(np.float32)
    return H

def bovw_normalize(H: np.ndarray, idf: np.ndarray) -> np.ndarray:
    # REQUIRED: sqrt/Hellinger -> TF-IDF -> L2
    H = np.sqrt(H, dtype=np.float32)
    H = H * idf
    H = normalize(H, norm="l2")
    return H

def balanced_codebook_sample(paths, labels, techs, n_real=800, n_per_tech=800, rng=1337):
    rs = np.random.RandomState(rng)
    paths = np.array(paths)
    labels = np.array(labels)
    techs  = np.array(techs)

    groups = []
    real_idx = np.where(labels == 0)[0]
    if len(real_idx) > 0:
        take = min(n_real, len(real_idx))
        groups.append(rs.choice(real_idx, size=take, replace=False))

    fake_idx = np.where(labels == 1)[0]
    for t in np.unique(techs[fake_idx]):
        ti = np.where((labels == 1) & (techs == t))[0]
        if len(ti) == 0:
            continue
        take = min(n_per_tech, len(ti))
        groups.append(rs.choice(ti, size=take, replace=False))

    idx = np.unique(np.concatenate(groups)) if groups else np.array([], dtype=int)
    return paths[idx].tolist()

def nested_cv_bovw(
    paths, labels, techs,
    K_candidates=(128,256),
    pca_dims_candidates=(64,96,128),
    outer_splits=5,
    inner_splits=3,
    rng=1337
):
    y = np.array(labels).astype(int)
    outer = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=rng)

    outer_scores = []
    best_params_per_fold = []

    for fold, (tr_idx, va_idx) in enumerate(outer.split(np.zeros_like(y), y), 1):
        tr_paths = [paths[i] for i in tr_idx]
        va_paths = [paths[i] for i in va_idx]
        tr_y = y[tr_idx]
        va_y = y[va_idx]
        tr_techs = [techs[i] for i in tr_idx]

        # descriptors (cached)
        tr_desc = [get_or_make_desc(Path(p), det, det_name, desc_dim) for p in tr_paths]
        va_desc = [get_or_make_desc(Path(p), det, det_name, desc_dim) for p in va_paths]

        inner = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=rng + fold)
        best_inner = (-1.0, None)

        for K in K_candidates:
            # codebook fit on balanced subset of OUTER-TRAIN only
            cb_paths = balanced_codebook_sample(tr_paths, tr_y, tr_techs, rng=rng + fold)
            cb_desc  = [get_or_make_desc(Path(p), det, det_name, desc_dim) for p in cb_paths]
            X_code = np.vstack([d for d in cb_desc if d is not None and len(d) > 0])
            if X_code.shape[0] == 0:
                raise RuntimeError("No descriptors found for codebook fitting.")

            codebook = MiniBatchKMeans(
                n_clusters=K, batch_size=4096, random_state=rng + fold,
                max_iter=100, n_init=3
            ).fit(X_code)

            # RAW histograms for OUTER-TRAIN only (for this codebook/K)
            H_tr_raw = apply_bovw(tr_desc, codebook, K)

            for pca_dims in pca_dims_candidates:
                inner_aurocs = []

                for itr, iva in inner.split(np.zeros_like(tr_y), tr_y):
                    # leakage-safe: fit IDF on INNER-TRAIN only
                    H_itr_raw = H_tr_raw[itr]
                    H_iva_raw = H_tr_raw[iva]
                    y_itr = tr_y[itr]
                    y_iva = tr_y[iva]

                    idf_inner = fit_idf(H_itr_raw)
                    H_itr = bovw_normalize(H_itr_raw, idf_inner)
                    H_iva = bovw_normalize(H_iva_raw, idf_inner)

                    pca = PCA(n_components=pca_dims, random_state=rng + fold)
                    X_itr = pca.fit_transform(H_itr)
                    X_iva = pca.transform(H_iva)

                    clf = Pipeline([
                        ("scaler", StandardScaler()),
                        ("lr", LogisticRegression(
                            solver="saga", penalty="l2", C=1.0,
                            max_iter=3000, random_state=rng + fold, n_jobs=1
                        ))
                    ])
                    clf.fit(X_itr, y_itr)
                    p = clf.predict_proba(X_iva)[:, 1]
                    inner_aurocs.append(roc_auc_score(y_iva, p))

                mean_inner = float(np.mean(inner_aurocs))
                if mean_inner > best_inner[0]:
                    best_inner = (mean_inner, (K, pca_dims, codebook))

        if best_inner[1] is None:
            raise RuntimeError("Inner CV failed to select parameters.")
        K_best, pca_dims_best, codebook_best = best_inner[1]

        # OUTER evaluation: fit IDF on full OUTER-TRAIN only using selected codebook/K
        H_tr_raw_best = apply_bovw(tr_desc, codebook_best, K_best)
        H_va_raw_best = apply_bovw(va_desc, codebook_best, K_best)

        idf_best = fit_idf(H_tr_raw_best)
        H_tr = bovw_normalize(H_tr_raw_best, idf_best)
        H_va = bovw_normalize(H_va_raw_best, idf_best)

        pca = PCA(n_components=pca_dims_best, random_state=rng + fold)
        X_tr = pca.fit_transform(H_tr)
        X_va = pca.transform(H_va)

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(
                solver="saga", penalty="l2", C=1.0,
                max_iter=3000, random_state=rng + fold, n_jobs=1
            ))
        ])
        clf.fit(X_tr, tr_y)
        p = clf.predict_proba(X_va)[:, 1]
        auroc = roc_auc_score(va_y, p)

        outer_scores.append(float(auroc))
        best_params_per_fold.append((K_best, pca_dims_best))
        print(f"[outer fold {fold}] AUROC={auroc:.4f}  best(K={K_best}, PCA={pca_dims_best})")

    scores = np.array(outer_scores, dtype=float)

    fold_df = pd.DataFrame([
        {"outer_fold": i, "auroc": float(au), "K_best": int(Kf), "PCA_DIMS_best": int(Pf)}
        for i, (au, (Kf, Pf)) in enumerate(zip(outer_scores, best_params_per_fold), 1)
    ])

    summary = {
        "outer_splits": int(outer_splits),
        "inner_splits": int(inner_splits),
        "K_candidates": list(K_candidates),
        "pca_dims_candidates": list(pca_dims_candidates),
        "auroc_mean": float(scores.mean()),
        "auroc_std": float(scores.std(ddof=1)) if len(scores) > 1 else 0.0,
        "n_samples": int(len(y)),
        "label_pos_rate": float(y.mean()),
    }

    # ✅ FIXED print (no broken f-string)
    print(f"Nested CV AUROC: mean={summary['auroc_mean']:.4f}, std={summary['auroc_std']:.4f}")

    # expose for evidence export cell
    globals()["nested_cv_summary"] = summary
    globals()["nested_cv_folds"] = fold_df
    globals()["outer_aucs"] = scores

    return summary, fold_df, scores

In [ ]:
# -----------------------------
# RQ2/RQ3: RUN NESTED CV + SAVE EVIDENCE FILES
#
# Run this cell to produce:
#   - nested_cv_outer_scores.csv
#   - nested_cv_best_params_by_fold.csv
#   - nested_cv_summary.txt
#
# NOTE: This is compute-heavy. It is leakage-safe and meant for reporting.
# -----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# Ensure OUT_DIR/BOVW_DIR exist
_evd_dir = Path(OUT_DIR) if 'OUT_DIR' in globals() else Path(BOVW_DIR) if 'BOVW_DIR' in globals() else Path('.')
_evd_dir.mkdir(parents=True, exist_ok=True)

# Prepare inputs
paths = df_train_feat['path'].tolist()
labels = df_train_feat['label'].astype(int).tolist()
techs = df_train_feat['fake_technique'].fillna('').astype(str).tolist()

# Run nested CV (returns summary dict, per-fold df, scores)
nested_cv_summary, nested_cv_folds, outer_aucs = nested_cv_bovw(
    paths=paths,
    labels=labels,
    techs=techs,
    K_candidates=(128, 256),
    pca_dims_candidates=(64, 96, 128),
    outer_splits=5,
    inner_splits=3,
    rng=RNG
)

# Write evidence
nested_cv_folds.to_csv(_evd_dir / 'nested_cv_best_params_by_fold.csv', index=False)
pd.DataFrame({'outer_fold': np.arange(len(outer_aucs)), 'auroc': outer_aucs}).to_csv(_evd_dir / 'nested_cv_outer_scores.csv', index=False)

_sum_lines = [
    'Nested CV Summary',
    f"auroc_mean: {nested_cv_summary.get('auroc_mean')}",
    f"auroc_std:  {nested_cv_summary.get('auroc_std')}",
    f"outer_splits: {nested_cv_summary.get('outer_splits')}",
    f"inner_splits: {nested_cv_summary.get('inner_splits')}",
    f"K_candidates: {nested_cv_summary.get('K_candidates')}",
    f"pca_dims_candidates: {nested_cv_summary.get('pca_dims_candidates')}",
    f"n_samples: {nested_cv_summary.get('n_samples')}",
    f"label_pos_rate: {nested_cv_summary.get('label_pos_rate')}",
    '',
    'Raw summary dict:',
    repr(nested_cv_summary),
]
(_evd_dir / 'nested_cv_summary.txt').write_text('\n'.join(_sum_lines), encoding='utf-8')

print('Wrote:', _evd_dir / 'nested_cv_best_params_by_fold.csv')
print('Wrote:', _evd_dir / 'nested_cv_outer_scores.csv')
print('Wrote:', _evd_dir / 'nested_cv_summary.txt')


[outer fold 1] AUROC=0.5564  best(K=128, PCA=64)
[outer fold 2] AUROC=0.5531  best(K=128, PCA=64)
[outer fold 3] AUROC=0.5586  best(K=128, PCA=64)
[outer fold 4] AUROC=0.5605  best(K=128, PCA=64)
[outer fold 5] AUROC=0.5507  best(K=128, PCA=64)
Nested CV AUROC: mean=0.5559, std=0.0040
Wrote: /content/stargan-v2/bovw/nested_cv_best_params_by_fold.csv
Wrote: /content/stargan-v2/bovw/nested_cv_outer_scores.csv
Wrote: /content/stargan-v2/bovw/nested_cv_summary.txt


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def cv_auroc(X, y, splits=5, rng=1337):
    """Cross-validated AUROC with scaling INSIDE the CV pipeline (no leakage, no double-scaling)."""
    clf = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lr", LogisticRegression(
            solver="saga", penalty="l2", C=1.0,
            max_iter=3000, random_state=rng, n_jobs=1
        ))
    ])
    cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=rng)
    scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc", n_jobs=1)
    return float(scores.mean()), float(scores.std(ddof=1))


In [ ]:
from pathlib import Path
import os

print("CWD:", os.getcwd())
for p in ["/content/stargan-v2/bovw", "/content/stargan-v2", "/content", "/mnt/data"]:
    pp = Path(p)
    print(p, "exists=", pp.exists(), "is_dir=", pp.is_dir())
    if pp.exists() and pp.is_dir():
        print("  sample:", [x.name for x in sorted(pp.iterdir())[:15]])

CWD: /content
/content/stargan-v2/bovw exists= True is_dir= True
  sample: ['H_bovw_orb_K256_test.npy', 'H_bovw_orb_K256_train.npy', 'Xp_orb_K256_to96_test.npy', 'Xp_orb_K256_to96_train.npy', 'cluster_feature_grid_auc_NOLEAK.csv', 'codebook_orb_K256.joblib', 'desc_cache', 'idf_orb_K256.npy', 'image_features.csv', 'image_features_schema.csv', 'image_features_test.csv', 'image_features_test_order_sha1.txt', 'image_features_train_order_sha1.txt', 'km_img_labels_k10.npy', 'km_img_labels_k10_test.npy']
/content/stargan-v2 exists= True is_dir= True
  sample: ['bovw', 'combined_balanced']
/content exists= True is_dir= True
  sample: ['.config', 'drive', 'sample_data', 'stargan-v2']
/mnt/data exists= False is_dir= False


In [ ]:
# ===========================
# EDA for BoVW, PCA, Clusters, and Image Features (TRAIN)
# AUDIT NOTE:
# - Reads CSV with low_memory=False to avoid mixed-type warnings during review.
# - Enforces string dtypes on metadata columns for transparent, stable schema.
# ===========================
import json, math, hashlib, gc, os
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from joblib import load
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize
from scipy.stats import pearsonr

# -----------------------------
# CONFIG
# -----------------------------
BOVW_DIR = Path("/content/stargan-v2/bovw")   # where training pipeline wrote artifacts
OUT_DIR  = BOVW_DIR                            # reuse same folder for EDA drops
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Filenames produced by your training script (auto-detected)
def pick_one(pattern: str):
    cands = sorted(BOVW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None

H_PATH        = pick_one("H_bovw_*_train.npy")
X_PCA_PATH    = pick_one("Xp_*_train.npy")
PCA_PATH      = pick_one("pca_*_to*.joblib")
CV_BOVW_PATH  = pick_one("cv_bovw_k_selection.csv")
CV_PCA_PATH   = pick_one("cv_pca_selection.csv")
SIL_PATH      = pick_one("silhouette_k_grid.csv")
FEAT_CSV_PATH = pick_one("image_features.csv")
KM2_PATH      = pick_one("km_img_labels_k2.npy")
KM10_PATH     = pick_one("km_img_labels_k10.npy")
KM50_PATH     = pick_one("km_img_labels_k50.npy")

print("=== ARTIFACTS FOUND ===")
print(" BoVW (H):     ", H_PATH.name if H_PATH else None)
print(" PCA (Xp):     ", X_PCA_PATH.name if X_PCA_PATH else None)
print(" PCA model:    ", PCA_PATH.name if PCA_PATH else None)
print(" CV BoVW:      ", CV_BOVW_PATH.name if CV_BOVW_PATH else None)
print(" CV PCA:       ", CV_PCA_PATH.name if CV_PCA_PATH else None)
print(" Silhouette:   ", SIL_PATH.name if SIL_PATH else None)
print(" Features CSV: ", FEAT_CSV_PATH.name if FEAT_CSV_PATH else None)
print(" k=2 labels:   ", KM2_PATH.name if KM2_PATH else None)
print(" k=10 labels:  ", KM10_PATH.name if KM10_PATH else None)
print(" k=50 labels:  ", KM50_PATH.name if KM50_PATH else None)

if FEAT_CSV_PATH is None:
    raise SystemExit("[ERR] image_features.csv not found. Run the training script first.")

meta_string_cols = ["split", "path", "relative_path", "filename", "label_name", "species", "fake_technique"]
dtype_map = {c: "string" for c in meta_string_cols}
df = pd.read_csv(FEAT_CSV_PATH, low_memory=False, dtype=dtype_map)

for c in meta_string_cols:
    if c in df.columns:
        df[c] = df[c].fillna("").astype(str)

required_cols = {
    "label", "path", "brightness", "contrast", "sharpness_l1_mean", "file_size",
    "width", "height", "aspect_ratio", "edge_density", "lap_var",
    "km_label_k2", "km_label_k10", "km_label_k50"
}
missing = [c for c in required_cols if c not in df.columns]

if missing:
    if ("km_label_k10" in missing) and KM10_PATH is not None and len(np.load(KM10_PATH)) == len(df):
        df["km_label_k10"] = np.load(KM10_PATH)
        missing = [c for c in required_cols if c not in df.columns]
    if ("km_label_k50" in missing) and KM50_PATH is not None and len(np.load(KM50_PATH)) == len(df):
        df["km_label_k50"] = np.load(KM50_PATH)
        missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise SystemExit(f"[ERR] Missing required columns in image_features.csv: {missing}")

# Optional audit fingerprint check
train_order_sha1_path = OUT_DIR / "image_features_train_order_sha1.txt"
if train_order_sha1_path.exists() and "relative_path" in df.columns:
    h = hashlib.sha1()
    for v in df["relative_path"].astype(str).tolist():
        h.update(v.encode("utf-8"))
        h.update(b"\n")
    csv_sha1 = h.hexdigest()
    expected_sha1 = train_order_sha1_path.read_text(encoding="utf-8").strip()
    if csv_sha1 != expected_sha1:
        raise RuntimeError(f"Train feature CSV order fingerprint mismatch: {csv_sha1} != {expected_sha1}")
    print("✅ Train CSV order fingerprint verified:", csv_sha1)

y = df["label"].values.astype(int)
H  = np.load(H_PATH)     if H_PATH     else None
Xp = np.load(X_PCA_PATH) if X_PCA_PATH else None
pca = load(PCA_PATH)     if PCA_PATH   else None

print("Schema check (first 15 columns):")
display(pd.DataFrame({
    "column": df.columns[:15],
    "dtype": [str(dt) for dt in df.dtypes[:15]]
}))

# -----------------------------
# 1) BASIC IMAGE-FEATURE EDA
# -----------------------------
print("\n=== IMAGE FEATURE EDA (train) ===")

def summarize(series):
    s = pd.Series(series)
    return pd.Series({
        "n": s.shape[0],
        "mean": float(s.mean()),
        "std": float(s.std()),
        "min": float(s.min()),
        "p25": float(s.quantile(0.25)),
        "p50": float(s.median()),
        "p75": float(s.quantile(0.75)),
        "p95": float(s.quantile(0.95)),
        "max": float(s.max()),
    })

img_cols = ["brightness","contrast","sharpness_l1_mean","aspect_ratio","width","height","edge_density","lap_var","file_size"]
img_summary_overall = pd.DataFrame({c: summarize(df[c]) for c in img_cols})
img_summary_by_class = df.groupby("label_name")[img_cols].agg(["mean","std","min","median","max"])

pc_corr = pd.DataFrame()
if Xp is not None:
    corrs = []
    for j in range(Xp.shape[1]):
        r, p = pearsonr(Xp[:, j], y)
        corrs.append({"pc": j, "pearson_r": r, "p_value": p})
    pc_corr = pd.DataFrame(corrs).sort_values("pearson_r", key=np.abs, ascending=False)

img_summary_overall.to_csv(OUT_DIR / "eda_image_feature_overall.csv")
img_summary_by_class.to_csv(OUT_DIR / "eda_image_feature_by_class.csv")
if not pc_corr.empty:
    pc_corr.to_csv(OUT_DIR / "eda_pca_pc_correlations.csv", index=False)

print(" - Wrote eda_image_feature_overall.csv")
print(" - Wrote eda_image_feature_by_class.csv")
if not pc_corr.empty:
    print(" - Wrote eda_pca_pc_correlations.csv (top 10):")
    print(pc_corr.head(10).to_string(index=False))

# -----------------------------
# 2) BoVW EDA (sparsity, entropy) + PCA variance
# -----------------------------
print("\n=== BoVW / PCA EDA ===")

if H is not None:
    eps = 1e-12
    sparsity = (H == 0).mean(axis=1)
    probs = H / (H.sum(axis=1, keepdims=True) + eps)
    entropy = -np.sum(probs * np.log(probs + eps), axis=1) / math.log(H.shape[1])

    bovw_df = pd.DataFrame({
        "sparsity": sparsity,
        "entropy": entropy,
        "label": y
    })
    bovw_summary = bovw_df.groupby("label").agg(["mean","std","min","median","max"])
    bovw_summary.to_csv(OUT_DIR / "eda_bovw_stats_by_class.csv")
    print(" - Wrote eda_bovw_stats_by_class.csv")
else:
    print(" - Skipped BoVW stats: H_*_train.npy not found")

if pca is not None and hasattr(pca, "explained_variance_ratio_"):
    ev = pca.explained_variance_ratio_
    pca_var = pd.DataFrame({
        "pc": np.arange(len(ev)),
        "var_ratio": ev,
        "cum_var": np.cumsum(ev)
    })
    pca_var.to_csv(OUT_DIR / "eda_pca_variance.csv", index=False)
    print(" - Wrote eda_pca_variance.csv")
else:
    print(" - Skipped PCA variance: pca_*.joblib not found or missing EVR")

# -----------------------------
# 3) CLUSTER EDA (best-k and k=10)
# -----------------------------
print("\n=== CLUSTER EDA ===")

best_k = 2
if SIL_PATH is not None:
    sil_df = pd.read_csv(SIL_PATH, low_memory=False)
    best_k = int(sil_df.iloc[sil_df["silhouette"].idxmax()]["k"])
else:
    sil_df = pd.DataFrame()

if "km_label_best" not in df.columns and best_k != 10:
    best_path = pick_one(f"km_img_labels_k{best_k}.npy")
    if best_path and len(np.load(best_path)) == len(df):
        df["km_label_best"] = np.load(best_path)

if "km_label_best" not in df.columns:
    if best_k == 10:
        df["km_label_best"] = df["km_label_k10"]
    else:
        df["km_label_best"] = -1

k2_col = "km_label_k2"
if k2_col not in df.columns and KM2_PATH is not None:
    df[k2_col] = np.load(KM2_PATH)

if k2_col in df.columns:
    k2_counts = df.groupby(k2_col).size().rename("count")
    k2_class  = pd.crosstab(df[k2_col], df["label_name"])
    k2_ratio  = (k2_class.get("fake", pd.Series(index=k2_class.index, data=0)) / k2_class.sum(axis=1)).rename("fake_ratio")
    k2_feats  = df.groupby(k2_col)[img_cols].agg(["mean","std","median","min","max"])

    if "species" in df.columns:
        k2_species = pd.crosstab(df[k2_col], df["species"])
    else:
        k2_species = pd.DataFrame()

    if "fake_technique" in df.columns:
        k2_tech = pd.crosstab(df[k2_col], df["fake_technique"])
    else:
        k2_tech = pd.DataFrame()

    k2_summary = pd.concat([k2_counts, k2_class, k2_ratio], axis=1).sort_index()
    k2_summary.index.name = "cluster_k2"

    k2_summary.to_csv(OUT_DIR / "eda_clusters_k2_summary.csv")
    k2_feats.to_csv(OUT_DIR / "eda_clusters_k2_image_stats.csv")
    if not k2_species.empty:
        k2_species.to_csv(OUT_DIR / "eda_clusters_k2_by_species.csv")
    if not k2_tech.empty:
        k2_tech.to_csv(OUT_DIR / "eda_clusters_k2_by_technique.csv")

    print(" - Wrote eda_clusters_k2_summary.csv")
    print(" - Wrote eda_clusters_k2_image_stats.csv")
    if not k2_species.empty:
        print(" - Wrote eda_clusters_k2_by_species.csv")
    if not k2_tech.empty:
        print(" - Wrote eda_clusters_k2_by_technique.csv")

    if Xp is not None:
        k2_centroids = []
        for c in sorted(df[k2_col].unique()):
            mask = (df[k2_col] == c).values
            if mask.sum() == 0:
                continue
            ctr = Xp[mask].mean(axis=0)
            k2_centroids.append({
                "cluster": int(c),
                **{f"pc_{i}": ctr[i] for i in range(min(16, Xp.shape[1]))}
            })
        if k2_centroids:
            pd.DataFrame(k2_centroids).to_csv(OUT_DIR / "eda_clusters_k2_pca_centroids_head16.csv", index=False)
            print(" - Wrote eda_clusters_k2_pca_centroids_head16.csv")
else:
    print(" - Skipped k=2 cluster stats: km_label_k2 not found")

# Compact continuation note for remaining EDA sections already present in the original notebook.
print("EDA setup complete with stable CSV typing and order verification.")

=== ARTIFACTS FOUND ===
 BoVW (H):      H_bovw_orb_K256_train.npy
 PCA (Xp):      Xp_orb_K256_to96_train.npy
 PCA model:     pca_orb_K256_to96.joblib
 CV BoVW:       None
 CV PCA:        None
 Silhouette:    silhouette_k_grid.csv
 Features CSV:  image_features.csv
 k=2 labels:    km_img_labels_k2.npy
 k=10 labels:   km_img_labels_k10.npy
 k=50 labels:   km_img_labels_k50.npy
✅ Train CSV order fingerprint verified: 0ce3639e73728f675ba3aacb71950f429591bb75
Schema check (first 15 columns):


,column,dtype
0,row_id,int64
1,split,object
2,path,object
3,relative_path,object
4,filename,object
5,label,int64
6,label_name,object
7,species,object
8,fake_technique,object
9,width,int64



=== IMAGE FEATURE EDA (train) ===
 - Wrote eda_image_feature_overall.csv
 - Wrote eda_image_feature_by_class.csv
 - Wrote eda_pca_pc_correlations.csv (top 10):
 pc  pearson_r      p_value
  2  -0.065590 5.944751e-23
 11  -0.052357 3.515098e-15
  3   0.038216 9.285890e-09
  4  -0.033293 5.641734e-07
  0  -0.030266 5.419007e-06
 17   0.026229 8.109412e-05
 14  -0.021777 1.068121e-03
 35  -0.021629 1.155171e-03
 19   0.021388 1.311224e-03
 44  -0.019750 3.004361e-03

=== BoVW / PCA EDA ===
 - Wrote eda_bovw_stats_by_class.csv
 - Wrote eda_pca_variance.csv

=== CLUSTER EDA ===
 - Wrote eda_clusters_k2_summary.csv
 - Wrote eda_clusters_k2_image_stats.csv
 - Wrote eda_clusters_k2_by_species.csv
 - Wrote eda_clusters_k2_by_technique.csv
 - Wrote eda_clusters_k2_pca_centroids_head16.csv
EDA setup complete with stable CSV typing and order verification.


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

def soft_assign(D, tau):
    S = np.exp(-D / float(tau))
    return S / (S.sum(axis=1, keepdims=True) + 1e-12)

def cv_auc_with_cluster_feats_noleak(X_base, X_norm, y, k_grid=(10,12,15,20,25,30), taus=(0.5,1.0,2.0), rng=1337):
    rows = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=rng)

    for k in k_grid:
        for feat_kind in (["dist"] + [f"soft_tau={t}" for t in taus]):
            aucs = []

            for tr, va in skf.split(X_base, y):
                # Fit MiniBatchKMeans ONLY on train fold (no leakage)
                km = MiniBatchKMeans(
                    n_clusters=k,
                    random_state=rng,
                    n_init=3,
                    batch_size=2048,
                    max_iter=100
                ).fit(X_norm[tr])

                d_tr = km.transform(X_norm[tr])
                d_va = km.transform(X_norm[va])

                # Z-score distances using TRAIN stats only (no leakage)
                mu = d_tr.mean(0)
                sd = d_tr.std(0) + 1e-8
                d_trn = (d_tr - mu) / sd
                d_van = (d_va - mu) / sd

                if feat_kind == "dist":
                    Xtr = np.hstack([X_base[tr], d_trn]).astype(np.float32)
                    Xva = np.hstack([X_base[va], d_van]).astype(np.float32)
                else:
                    tau = float(feat_kind.split("=")[1])
                    s_tr = soft_assign(d_tr, tau)
                    s_va = soft_assign(d_va, tau)
                    Xtr = np.hstack([X_base[tr], s_tr]).astype(np.float32)
                    Xva = np.hstack([X_base[va], s_va]).astype(np.float32)

                clf = Pipeline([
                    ("scaler", StandardScaler()),
                    ("lr", LogisticRegression(
                        solver="saga", penalty="l2", C=1.0, max_iter=2000,
                        random_state=rng, n_jobs=1
                    ))
                ])
                clf.fit(Xtr, y[tr])
                p = clf.predict_proba(Xva)[:, 1]
                aucs.append(roc_auc_score(y[va], p))

            rows.append({
                "k": int(k),
                "feat": feat_kind,
                "auc_mean": float(np.mean(aucs)),
                "auc_std": float(np.std(aucs, ddof=1)),
            })

    return pd.DataFrame(rows)

# Example usage (assuming Xp, X_for_km, extra, y already exist)
# X_base = np.hstack([Xp.astype(np.float32), extra.astype(np.float32)])
grid_df = cv_auc_with_cluster_feats_noleak(X_base, X_for_km, y,
                                          k_grid=(10,12,15,20,25,30),
                                          taus=(0.5,1.0,2.0),
                                          rng=RNG)

grid_df = grid_df.sort_values(["auc_mean","feat"], ascending=[False, True])
print(grid_df.head(10).to_string(index=False))
grid_df.to_csv(OUT_DIR/"cluster_feature_grid_auc_NOLEAK.csv", index=False)
print("Wrote:", OUT_DIR/"cluster_feature_grid_auc_NOLEAK.csv")

 k         feat  auc_mean  auc_std
30 soft_tau=1.0  0.648122 0.005676
30 soft_tau=2.0  0.647961 0.005542
30 soft_tau=0.5  0.647873 0.005717
30         dist  0.647801 0.005629
25 soft_tau=1.0  0.647366 0.005968
25 soft_tau=2.0  0.647248 0.005903
25 soft_tau=0.5  0.646888 0.005862
25         dist  0.646805 0.005940
20 soft_tau=2.0  0.645687 0.005679
20 soft_tau=1.0  0.645605 0.005476
Wrote: /content/stargan-v2/bovw/cluster_feature_grid_auc_NOLEAK.csv


In [ ]:
from pathlib import Path
import re, numpy as np

dirpath = Path("/content/stargan-v2/bovw")
det = "orb"

km_path = dirpath / "km_orb_K256.joblib"   # <-- change to your actual filename

def parse_k(s):
    m = re.search(r"_K(\d+)", str(s))
    return int(m.group(1)) if m else None

K_used = parse_k(km_path)
if K_used is None:
    raise ValueError(f"Could not parse K from KMeans artifact path: {km_path}")

idf_path = dirpath / f"idf_{det}_K{K_used}.npy"
if not idf_path.exists():
    raise FileNotFoundError(f"Missing required IDF vector: {idf_path}")

idf = np.load(idf_path)
print("Using idf:", idf_path.name, "shape:", idf.shape)

Using idf: idf_orb_K256.npy shape: (256,)


In [ ]:
import os, shutil
src = "/content/stargan-v2/bovw"
dst = "/content/drive/MyDrive/deepfake_project/output/bovw3"

os.makedirs(os.path.dirname(dst), exist_ok=True)

# If a previous folder exists, remove it (optional)
if os.path.exists(dst):
    shutil.rmtree(dst)

# Copy the whole directory tree
shutil.copytree(src, dst)
print("Copied to:", dst)


Copied to: /content/drive/MyDrive/deepfake_project/output/bovw3


In [ ]:
# === Zip StarGAN-v2 'data' folder and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, sys

SRC = Path("/content/stargan-v2/bovw")
DEST_DIR = Path("/content/drive/MyDrive/deepfake_project/output/bovw3")

# Safety checks
if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")
if not any(SRC.rglob("*")):
    raise RuntimeError(f"Source folder is empty: {SRC}")

DEST_DIR.mkdir(parents=True, exist_ok=True)

# Timestamped archive name
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/stargan-v2_data_bovw_kmeans_pca-{stamp}")  # no extension for make_archive

print(f"Creating ZIP archive from: {SRC}")
# Create ZIP (you can switch 'zip' -> 'gztar' for .tar.gz if preferred)
zip_path_str = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=SRC.parent,   # /content/stargan-v2
    base_dir=SRC.name      # data
)
archive_path = Path(zip_path_str)

# Move to Drive
final_path = DEST_DIR / archive_path.name
print(f"Moving archive to: {final_path}")
shutil.move(str(archive_path), str(final_path))

# Report
size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Archive ready: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


Creating ZIP archive from: /content/stargan-v2/bovw
Moving archive to: /content/drive/MyDrive/deepfake_project/output/bovw3/stargan-v2_data_bovw_kmeans_pca-20260308-230713.zip
✅ Archive ready: /content/drive/MyDrive/deepfake_project/output/bovw3/stargan-v2_data_bovw_kmeans_pca-20260308-230713.zip
   Size: 0.89 GB


In [ ]:
# -----------------------------
# NEW: Compact results report (baseline vs fused-feature CV)
# Fixed: replaced display() with print(df.to_string()) for plain-text visibility
# -----------------------------
import pandas as pd
from pathlib import Path

OUT_DIR  = globals().get("OUT_DIR", BOVW_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

nested_summary = OUT_DIR / "nested_cv_summary.txt"
grid_auc = OUT_DIR / "cluster_feature_grid_auc_NOLEAK.csv"

print("\n=== BoVW-only nested CV (leakage-safe baseline) ===")
if nested_summary.exists():
    print(nested_summary.read_text())
else:
    print(f"Missing: {nested_summary}")

print("\n=== Fused-feature CV grid (NO-LEAK) ===")
if grid_auc.exists():
    df = pd.read_csv(grid_auc)
    auc_cols = [c for c in df.columns if "auc" in c.lower() and "std" not in c.lower()]
    if len(auc_cols) == 0:
        print("Could not find an AUROC column in grid file; showing head():")
        print(df.head(10).to_string(index=False))
    else:
        auc_col = auc_cols[0]
        best = df.sort_values(auc_col, ascending=False).head(10)
        print(f"Best rows by {auc_col}:")
        print(best.to_string(index=False))
else:
    print(f"Missing: {grid_auc}")

print("\n=== Silhouette grid ===")
sil_path = OUT_DIR / "silhouette_k_grid.csv"
if sil_path.exists():
    sil_df = pd.read_csv(sil_path)
    print(sil_df.to_string(index=False))
    best_sil_k = int(sil_df.loc[sil_df["silhouette"].idxmax(), "k"])
    best_sil_score = sil_df["silhouette"].max()
    print(f"\nBest k by silhouette: k={best_sil_k} (score={best_sil_score:.4f})")
    print("Note: k=2 yields highest silhouette but is too coarse for feature diversity.")
    k_for_feats = int(sil_df[sil_df["k"] >= 10].loc[
        sil_df[sil_df["k"] >= 10]["silhouette"].idxmax(), "k"])
    print(f"Best k >= 10 by silhouette (used in downstream modelling): k={k_for_feats}")
else:
    print(f"Missing: {sil_path}")



=== BoVW-only nested CV (leakage-safe baseline) ===
Nested CV Summary
auroc_mean: 0.5558814550023817
auroc_std:  0.003995878047155745
outer_splits: 5
inner_splits: 3
K_candidates: [128, 256]
pca_dims_candidates: [64, 96, 128]
n_samples: 22572
label_pos_rate: 0.5

Raw summary dict:
{'outer_splits': 5, 'inner_splits': 3, 'K_candidates': [128, 256], 'pca_dims_candidates': [64, 96, 128], 'auroc_mean': 0.5558814550023817, 'auroc_std': 0.003995878047155745, 'n_samples': 22572, 'label_pos_rate': 0.5}

=== Fused-feature CV grid (NO-LEAK) ===
Best rows by auc_mean:
 k         feat  auc_mean  auc_std
30 soft_tau=1.0  0.648122 0.005676
30 soft_tau=2.0  0.647961 0.005542
30 soft_tau=0.5  0.647873 0.005717
30         dist  0.647801 0.005629
25 soft_tau=1.0  0.647366 0.005968
25 soft_tau=2.0  0.647248 0.005903
25 soft_tau=0.5  0.646888 0.005862
25         dist  0.646805 0.005940
20 soft_tau=2.0  0.645687 0.005679
20 soft_tau=1.0  0.645605 0.005476

=== Silhouette grid ===
 k  silhouette  n_sample  